# 9. Time-Window Selection for the Contextual Risk Index (Lisbon)

This notebook searches for a set of spatiotemporal scenarios (evaluation times) and a time-window
table that are coherent and that make the CRI differ between scenarios, without flattening its spatial
spread within each scenario.

Method:

1. Each evaluation time maps to an **activity state**: the set of active `(category, vi)` pairs.
   Vulnerability depends only on that state, so the week is scanned every 30 min (at :15/:45, away from
   window edges) and VERUS runs once per distinct state.
2. Each state runs on a **fresh `VERUS` instance**. Reusing one instance carries `vi` values over from the
   previous `run()` (it overwrites `poti_df`), which is what notebook 01 does for s1→s4.
3. Vulnerability is rescaled by the common maximum across the states of a table, then combined with the
   response, risk, Gini and income layers using `compute_cri` from notebook 08 (`gamma = 0.5`).
4. Every set of 4 states is scored by temporal distinctness (worst-pair and mean `1 − Spearman`, overlap of
   the top 10% priority hexagons), subject to spatial spread and vulnerability-weight floors.

Tables: `T0` is `default_time_windows.csv`; `T1`/`T2` are candidates defined in `tw_specs.py`
(`T1g`/`T2g` keep schools active only at the gates, as in T0).

Requires verus >= 1.1.1 (zero-baseline VL normalization, `value / max_vl`).
Vulnerability layers are cached in `../data/tw_analysis/<table>/`.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import verus
from verus.data import TimeWindowGenerator
from verus.grid import HexagonGridGenerator

import cri_utils as cu
import tw_specs

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)
print("verus", verus.__version__)

In [ ]:
city = "Lisbon"
out_dir = "../data/tw_analysis"
tables = ["T0", "T1", "T1g", "T2", "T2g"]
gamma = 0.5
min_categories = 3      # a scenario must activate at least 3 categories
w_vuln_floor = 0.15     # vulnerability must keep a meaningful weight in the CRI
top_frac = 0.10         # priority set = top 10% hexagons by CRI

# Current article scenarios (notebook 01)
current_times = {
    "s1": "2023-11-11 10:20:00",
    "s2": "2023-11-06 08:40:00",
    "s3": "2023-11-06 12:30:00",
    "s4": "2023-11-06 17:30:00",
}

## 9.1. Inputs and grid compatibility

In [ ]:
poti_df = pd.read_csv(f"../data/poti/{city.lower()}_dataset_buffered.csv")
grid = HexagonGridGenerator(
    region=f"../data/cities/{city.lower()}.geojson", edge_length=100, verbose=False
).run(add_random_values=False)
base = gpd.read_file(f"../data/multi_layers/{city}_multi_layer_all_scenarios.geojson").set_index("hex_id")

# The new grid must match the committed layers hexagon by hexagon
same_ids = set(grid["hex_id"]) == set(base.index)
g = grid.set_index("hex_id").to_crs(3763).centroid
b = base.loc[g.index].to_crs(3763).centroid
print(f"hexagons: grid={len(grid)} base={len(base)} same ids={same_ids} "
      f"max centroid offset={g.distance(b).max():.2e} m")
assert same_ids and g.distance(b).max() < 1e-6

## 9.2. Time-window tables

In [ ]:
tw = {"T0": pd.read_csv("../data/time_windows/default_time_windows.csv")}
for name in tables[1:]:
    tw[name] = cu.build_table(tw_specs.SPECS[name])
    tw[name].to_csv(f"{out_dir}/time_windows_{name}.csv", index=False)

for name, t in tw.items():
    st = cu.state_table(t)
    empty = int(st.loc[st["n_categories"] == 0, "n_slots"].sum())
    print(f"{name}: {len(t)} windows, overlaps within a category={len(cu.check_table(t))}, "
          f"distinct states={len(st)}, half-hour slots with no active category={empty}/336")

In [ ]:
# Changes of each candidate table against T0
t0 = cu.describe_table(cu.build_table(tw_specs.T0)).drop(columns="n_days")
for name in ["T1", "T2"]:
    d = cu.describe_table(tw[name]).drop(columns="n_days")
    diff = t0.merge(d, how="outer", indicator=True)
    diff["_merge"] = diff["_merge"].astype(str)
    diff = diff[diff["_merge"] != "both"].replace({"left_only": "T0 only", "right_only": f"{name} only"})
    print(f"--- T0 vs {name}")
    print(diff.sort_values(["category", "days", "window"]).to_string(index=False))

## 9.3. Vulnerability and CRI per activity state

In [ ]:
states, vuln, cri, cri_diag, metrics = {}, {}, {}, {}, {}
for name in tables:
    st, V = cu.compute_states(poti_df, tw[name], grid, f"{out_dir}/{name}")
    assert V.shape[0] == len(base) and not V.isna().any().any()
    vuln[name], cri[name], cri_diag[name], m = cu.cri_per_state(V, base, gamma=gamma)
    st = st.set_index("state_id")
    metrics[name] = m.join(st[["n_categories", "n_slots", "days", "first_hm", "last_hm", "label"]])
    states[name] = st
    assert cri[name].min().min() >= 0 and cri[name].max().max() <= 1
    metrics[name].to_csv(f"{out_dir}/{name}/state_metrics.csv")

metrics["T0"].drop(columns="label").round(3)

### Entropy weight of vulnerability

The entropy weighting in `compute_cri` gives less weight to indicators that are more uniform in space.
Busy scenarios spread vulnerability over the city, so vulnerability loses weight exactly when it is high.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for name, m in metrics.items():
    ax.scatter(m["vuln_mean"], m["w_vuln"], label=name, s=35)
ax.set_xlabel("Mean vulnerability (common scale)")
ax.set_ylabel("Entropy weight of vulnerability in the core score")
ax.axhline(w_vuln_floor, color="grey", ls="--", lw=1)
ax.legend()
plt.tight_layout()
plt.show()
print("Spearman(vuln_mean, w_vuln) per table:",
      {n: round(m["vuln_mean"].corr(m["w_vuln"], method="spearman"), 3) for n, m in metrics.items()})

## 9.4. Baseline: current scenarios s1–s4

In [ ]:
def state_at(name, time_str):
    lab = cu.state_label(cu.active_state(tw[name], TimeWindowGenerator.to_unix_epoch(time_str)))
    return states[name].reset_index().set_index("label").loc[lab, "state_id"]

committed = gpd.read_file(f"../data/multi_layers/{city}_multi_layer_all_scenarios_with_CRI.geojson").set_index("hex_id")
rows = [{"table": "committed (article)", **cu.set_summary(committed[[f"s{i}_cri" for i in range(1, 5)]], top_frac)}]
for name in tables:
    ids = [state_at(name, t) for t in current_times.values()]
    rows.append({"table": f"{name} current times {ids}", **cu.set_summary(cri[name][ids], top_frac)})
baseline = pd.DataFrame(rows).set_index("table")
baseline.round(4)

In [ ]:
# Pairwise detail for the article times under T0 (fresh assessor)
ids = [state_at("T0", t) for t in current_times.values()]
p = cu.pairwise_temporal(cri["T0"][ids], top_frac)
p[["a", "b"]] = p[["a", "b"]].replace(dict(zip(ids, current_times)))
p.round(4)

## 9.5. Search over sets of 4 scenarios

Candidate states activate at least `min_categories` categories and keep `w_vuln >= w_vuln_floor`.
Sets are ranked by the worst pair (`min_dissim = 1 − max Spearman`), which rules out redundant
scenarios, then by `mean_dissim`. The Pareto front also minimizes the largest top-10% overlap.

In [ ]:
results = []
for name in tables:
    m = metrics[name]
    cand = m[(m["n_categories"] >= min_categories) & (m["w_vuln"] >= w_vuln_floor)].index.tolist()
    r = cu.search_sets(cri[name], cand, k=4, frac=top_frac)
    r["table"] = name
    r["n_candidates"] = len(cand)
    r["min_cri_iqr"] = r["set"].map(lambda s: m.loc[list(s), "cri_iqr"].min())
    results.append(r)
results = pd.concat(results, ignore_index=True)
results.to_csv(f"{out_dir}/set_search.csv", index=False)

best = (results.sort_values(["min_dissim", "mean_dissim"], ascending=False)
        .groupby("table").head(3).sort_values(["table", "min_dissim"], ascending=[True, False]))
best.round(4)

In [ ]:
front = cu.pareto_front(results, ["min_dissim", "mean_dissim", "min_cri_iqr"], ["max_jaccard_top"])
front.sort_values("min_dissim", ascending=False).head(15).round(4)

In [ ]:
def describe_set(name, ids):
    m = metrics[name].loc[list(ids)]
    return m[["days", "first_hm", "last_hm", "n_slots", "label", "vuln_mean", "w_vuln", "cri_iqr"]]

top = best.sort_values(["min_dissim", "mean_dissim"], ascending=False).groupby("table").head(1)
for _, r in top.iterrows():
    print(f"=== {r['table']}: min_dissim={r['min_dissim']:.3f} mean_dissim={r['mean_dissim']:.3f} "
          f"max_jaccard_top={r['max_jaccard_top']:.3f}")
    print(describe_set(r["table"], r["set"]).round(3).to_string())

## 9.6. Maps of the best set per table

In [ ]:
geo = base[["geometry"]]
for _, r in top.iterrows():
    name, ids = r["table"], list(r["set"])
    fig, axes = plt.subplots(1, 4, figsize=(22, 6))
    for ax, sid in zip(axes, ids):
        m = metrics[name].loc[sid]
        geo.assign(cri=cri[name][sid]).plot(column="cri", cmap="RdYlGn_r", vmin=0, vmax=1, ax=ax)
        ax.set_title(f"{name} {sid}: {m['days']} {m['first_hm']}–{m['last_hm']}", fontsize=11)
        ax.set_axis_off()
    fig.suptitle(f"CRI, best set for {name}", fontsize=14)
    plt.tight_layout()
    fig.savefig(f"{out_dir}/{name}_best_set_cri.png", dpi=150, bbox_inches="tight")
    plt.show()

## 9.7. Interpretable scenario sets

The unconstrained search can pick adjacent states (e.g. 19:15 and 20:15), which are hard to name.
Here each set takes one scenario per activity regime and is evaluated in every table.

In [ ]:
regime_sets = {
    "article (s1-s4)": list(current_times.values()),
    "regimes: AM rush, late morning, weekday night, Saturday day": [
        "2023-11-06 08:30:00", "2023-11-06 11:00:00", "2023-11-06 21:00:00", "2023-11-11 11:00:00"],
    "regimes with PM rush instead of night": [
        "2023-11-06 08:30:00", "2023-11-06 11:00:00", "2023-11-06 17:30:00", "2023-11-11 11:00:00"],
}
rows = []
for name in tables:
    for set_name, times in regime_sets.items():
        ids = [state_at(name, t) for t in times]
        missing = [i for i in ids if i not in cri[name].columns]
        if missing:  # e.g. T0 has no active category on weekday nights
            rows.append({"table": name, "set": set_name, "states": ids,
                         "worst_pair": f"not evaluable: {missing} has no active category"})
            continue
        p = cu.pairwise_temporal(cri[name][ids], top_frac).sort_values("spearman", ascending=False).iloc[0]
        rows.append({"table": name, "set": set_name, "states": ids,
                     **cu.set_summary(cri[name][ids], top_frac),
                     "worst_pair": f"{p['a']}-{p['b']}",
                     "min_w_vuln": metrics[name].loc[ids, "w_vuln"].min()})
regimes = pd.DataFrame(rows)
regimes.to_csv(f"{out_dir}/regime_sets.csv", index=False)
regimes.round(3)

In [ ]:
# Recommended: table T1g (no new vi values) with one scenario per regime
rec_table = "T1g"
rec_times = regime_sets["regimes: AM rush, late morning, weekday night, Saturday day"]
rec_ids = [state_at(rec_table, t) for t in rec_times]
print(describe_set(rec_table, rec_ids).round(3).to_string())
print(cu.pairwise_temporal(cri[rec_table][rec_ids], top_frac).round(3).to_string())

fig, axes = plt.subplots(1, 4, figsize=(22, 6))
for ax, sid, t in zip(axes, rec_ids, rec_times):
    geo.assign(cri=cri[rec_table][sid]).plot(column="cri", cmap="RdYlGn_r", vmin=0, vmax=1, ax=ax)
    ax.set_title(f"{rec_table} {sid}: {pd.Timestamp(t):%a %H:%M}", fontsize=11)
    ax.set_axis_off()
fig.suptitle(f"CRI, recommended scenario set ({rec_table})", fontsize=14)
plt.tight_layout()
fig.savefig(f"{out_dir}/{rec_table}_recommended_set_cri.png", dpi=150, bbox_inches="tight")
plt.show()

## 9.8. Recommended scenario regimes

Recommended table: **T1g** (T0 with gaps filled by `vi` values already present in T0, schools active
only at the gates). Recommended scenarios, one per activity regime:

| Regime | Evaluation time | Why it is a distinct regime |
|---|---|---|
| Weekday morning rush | Mon 2023-11-06 08:30 | commuting peak: stations, universities, industry, hospitals, school gates |
| Weekday late morning | Mon 2023-11-06 11:00 | daytime occupancy: attractions, universities, hospitals at base level, off-peak transport |
| Weekday night | Mon 2023-11-06 21:00 | residual activity: hospitals, stations, bus stations |
| Saturday daytime | Sat 2023-11-11 11:00 | weekend leisure: attractions, malls, stations |

The evening rush (17:30) is left out on purpose: it is almost identical to the morning rush in every
table tested (Spearman ≈ 0.99), because the two peaks share the same `vi` values. This is the s2 ≈ s4
redundancy of the article scenarios.

The cell below writes the recommended set and its metrics to `../data/tw_analysis/recommended_regimes.csv`.

In [ ]:
regime_names = ["weekday morning rush", "weekday late morning", "weekday night", "saturday daytime"]
rec = describe_set(rec_table, rec_ids).reset_index()
rec.insert(0, "evaluation_time", rec_times)
rec.insert(0, "regime", regime_names)
rec.insert(0, "table", rec_table)
rec["cri_std"] = metrics[rec_table].loc[rec_ids, "cri_std"].to_numpy()
rec["cri_gvf5"] = metrics[rec_table].loc[rec_ids, "cri_gvf5"].to_numpy()
rec.to_csv(f"{out_dir}/recommended_regimes.csv", index=False)

summary = cu.set_summary(cri[rec_table][rec_ids], top_frac)
print("Set metrics:", {k: round(v, 3) for k, v in summary.items()})
rec.round(3)

## 9.9. Entropy-weighting variants

Entropy weighting is part of the core contribution and stays. The variants below change only the data
on which the Shannon entropy weights are computed:

- **E0**: per-scenario weights, as in notebook 08 (`compute_cri`).
- **E1**: weights computed once on the indicators pooled over the scenarios of the set; normalization and
  the final min-max use the pooled range, so all scenarios share one CRI scale.
- **E1w**: as E1, pooled over every activity state of the week (one set of weights per city and table,
  independent of the scenarios chosen).
- **E2**: per-scenario weights, vulnerability on the common scale instead of per-scenario min-max.
- **E3**: per-scenario weights with a floor on each core weight (0.20, 0.25, 0.30), renormalized.

With a single scenario, E1 equals E0, and a floor of 0 equals E0 (checked below). Under a shared scale
(E1, E1w) the IQR also reflects the CRI level of each scenario, so GVF is the fairer spatial measure.

In [ ]:
# Sanity checks against compute_cri (notebook 08)
for sid in rec_ids[:2]:
    p, _, _ = cu.cri_pooled(vuln[rec_table], base, [sid])
    f, _ = cu.cri_floor(vuln[rec_table], base, [sid], 0.0)
    assert (p[sid] - cri[rec_table][sid]).abs().max() < 1e-12
    assert (f[sid] - cri[rec_table][sid]).abs().max() < 1e-12
print("E1 on one scenario and E3 with floor 0 reproduce compute_cri")

def entropy_variants(name, ids):
    v = vuln[name]
    week = list(v.columns)
    m = metrics[name]
    out = {"E0 per-scenario (08)": (cri[name][ids], {s: m.loc[s, ["w_vuln", "w_resp", "w_risk"]].to_numpy() for s in ids})}
    p, w, _ = cu.cri_pooled(v, base, ids)
    out["E1 pooled over set"] = (p, {s: w for s in ids})
    p, w, _ = cu.cri_pooled(v, base, ids, pool=week)
    out["E1w pooled over week"] = (p, {s: w for s in ids})
    d = {s: cu.compute_cri_common_vuln(v[s], base["response_value"], base["risk_value"],
                                       base["gini_value"], base["income_value"], gamma=gamma) for s in ids}
    out["E2 common vuln scale"] = (pd.DataFrame({s: cu.normalize_values(d[s][0]) for s in ids}), {s: d[s][1] for s in ids})
    for fl in (0.20, 0.25, 0.30):
        out[f"E3 floor {fl:.2f}"] = cu.cri_floor(v, base, ids, fl, gamma=gamma)
    rows = []
    for variant, (c, w) in out.items():
        ss = cu.set_summary(c[ids], top_frac)
        rows.append({"variant": variant, **{k: ss[k] for k in ["min_dissim", "mean_dissim", "max_jaccard_top"]},
                     "min_gvf5": min(cu.gvf(c[s]) for s in ids),
                     "min_iqr": min(cu.spatial_metrics(c[s])["iqr"] for s in ids),
                     "w_vuln": " ".join(f"{w[s][0]:.2f}" for s in ids),
                     "min_rho_vs_E0": min(c[s].corr(cri[name][s], method="spearman") for s in ids)})
    return pd.DataFrame(rows)

variant_tables = []
for label, name, times in [("T1g regimes", rec_table, rec_times),
                           ("T0 article times", "T0", list(current_times.values()))]:
    ids = [state_at(name, t) for t in times]
    variant_tables.append(entropy_variants(name, ids).assign(case=label))
variants = pd.concat(variant_tables, ignore_index=True)
variants.to_csv(f"{out_dir}/entropy_variants.csv", index=False)
variants.round(3)

In [ ]:
# E1w: week-pooled weights for T1g, per-state CRI and the set search under E1w
v = vuln[rec_table]
week = list(v.columns)
cri_e1w, w_core, w_eq = cu.cri_pooled(v, base, week, pool=week)
print("E1w core weights (vuln, resp, risk):", np.round(w_core, 3), " equity (gini, income):", np.round(w_eq, 3))
per_state = pd.DataFrame({
    "E1w_mean": cri_e1w.mean(),
    "E1w_gvf5": [cu.gvf(cri_e1w[s]) for s in week],
    "rho_E1w_vs_E0": [cri_e1w[s].corr(cri[rec_table][s], method="spearman") for s in week],
}, index=week).join(metrics[rec_table][["days", "first_hm", "last_hm", "n_categories", "w_vuln"]])
print(per_state.round(3).to_string())

m = metrics[rec_table]
cand = m[m["n_categories"] >= min_categories].index.tolist()
search_e1w = cu.search_sets(cri_e1w, cand, k=4, frac=top_frac).sort_values(["min_dissim", "mean_dissim"], ascending=False)
search_e1w.to_csv(f"{out_dir}/set_search_T1g_E1w.csv", index=False)
rank = int((search_e1w["min_dissim"] > search_e1w.loc[search_e1w["set"] == tuple(rec_ids), "min_dissim"].iloc[0]).sum()) + 1
print(f"Recommended regimes under E1w: rank {rank} of {len(search_e1w)} by min_dissim")
search_e1w.head(6).round(3)

### Findings

- **E1w** gives the largest gain in temporal distinctness (T1g regimes: mean dissimilarity 0.277 → 0.373;
  article times: 0.206 → 0.407) and removes the collapse of the vulnerability weight in busy scenarios:
  entropy computed on the whole week gives vulnerability a single weight (≈0.57). Within-scenario class
  separation is kept (GVF 0.92–0.95). The CRI level now differs between scenarios (≈0.24 at night,
  ≈0.55 at peaks), which the per-scenario min-max of E0 hid.
- Cost of E1w: busy scenarios change the most relative to E0 (Spearman ≈0.69–0.75), so the article's
  CRI maps would change. The best scenario set also changes under E1w, so if E1w is adopted, the set
  should be chosen again under it.
- **E1** (pooled over the set) is a milder version; its weights depend on which scenarios are chosen.
- **E2** reduces contrast and changes the ranks the most: discarded.
- **E3** (floors) barely moves the result; floors high enough to matter (0.30) reduce temporal contrast.

## 9.10. Robustness of pooled entropy weights

For a tool in which the vulnerability model changes from time to time (new POTIs, edited windows), the
weighting must not depend on arbitrary choices. Two pooling schemes are compared:

- **E1w**: every distinct activity state of the week counts once.
- **E1wd**: every state counts by its fraction of the week (`cu.week_fractions`), through a
  frequency-weighted version of the same entropy function (`cu.compute_entropy_weights_weighted`,
  equal to `compute_entropy_weights` with unit weights and to row replication with integer weights).

Properties tested: (A) scan resolution, (B) splitting a state into two identical ones, (C) temporal and
spatial discrimination of the regimes, (D) response of the CRI when the time-window table changes.

In [ ]:
# Frequency-weighted entropy matches the original function
X = np.random.default_rng(1).random((500, 3)) ** 3
mult = np.random.default_rng(2).integers(1, 5, 500)
assert np.allclose(cu.compute_entropy_weights_weighted(X, np.ones(500))[0], cu.compute_entropy_weights(X)[0])
assert np.allclose(cu.compute_entropy_weights_weighted(X, mult)[0], cu.compute_entropy_weights(np.repeat(X, mult, axis=0))[0])

cand_tables = ["T1", "T1g", "T2", "T2g"]
reg_times = regime_sets["regimes: AM rush, late morning, weekday night, Saturday day"]
reg_ids = {n: [state_at(n, t) for t in reg_times] for n in cand_tables}

# (A) Weights per scheme and table; slot counts at 15/30/60-min scans are uniform rescalings
rows = []
for n in cand_tables:
    v, st = vuln[n], states[n]
    week = list(v.columns)
    slots = st["n_slots"].to_dict()
    schemes = {"E1w": {s: 1.0 for s in week},
               "slots 15 min": {s: 2.0 * slots[s] for s in week},
               "slots 30 min": {s: 1.0 * slots[s] for s in week},
               "slots 60 min": {s: 0.5 * slots[s] for s in week},
               "E1wd": cu.week_fractions(st)}
    for k, pw in schemes.items():
        _, w, we = cu.cri_pooled_weighted(v, base, week, pw)
        rows.append({"table": n, "scheme": k, "w_vuln": w[0], "w_resp": w[1], "w_risk": w[2], "w_gini": we[0]})
    m = metrics[n]
    rows.append({"table": n, "scheme": "E0 range", "w_vuln": f"{m.w_vuln.min():.3f}-{m.w_vuln.max():.3f}"})
weights_by_scheme = pd.DataFrame(rows)
weights_by_scheme.to_csv(f"{out_dir}/entropy_weights_by_scheme.csv", index=False)
weights_by_scheme.round(3)

In [ ]:
# (B) Split the longest T1g state into two identical states
v, st = vuln[rec_table], states[rec_table]
week = list(v.columns)
pi = cu.week_fractions(st)
big = max(pi, key=pi.get)
v_split = v.assign(**{big + "_b": v[big]})
_, w1, _ = cu.cri_pooled_weighted(v, base, week, {s: 1.0 for s in week})
_, w2, _ = cu.cri_pooled_weighted(v_split, base, week, {**{s: 1.0 for s in week}, big + "_b": 1.0})
_, w3, _ = cu.cri_pooled_weighted(v, base, week, pi)
_, w4, _ = cu.cri_pooled_weighted(v_split, base, week, {**pi, big: pi[big] / 2, big + "_b": pi[big] / 2})
print(f"Split {big} ({pi[big]:.0%} of the week)")
print("E1w  core weights before/after:", w1.round(4), w2.round(4))
print("E1wd core weights before/after:", w3.round(4), w4.round(4))
assert np.allclose(w3, w4)

In [ ]:
# (C) T1g regimes under E0, E1w and E1wd, and the set search under E1wd
cri_e1wd, w_e1wd, _ = cu.cri_pooled_weighted(v, base, week, pi)
cri_e1w_, _, _ = cu.cri_pooled_weighted(v, base, week, {s: 1.0 for s in week})
rows = []
for k, c in [("E0", cri[rec_table]), ("E1w", cri_e1w_), ("E1wd", cri_e1wd)]:
    ss = cu.set_summary(c[rec_ids], top_frac)
    rows.append({"variant": k, **{x: ss[x] for x in ["min_dissim", "mean_dissim", "max_jaccard_top"]},
                 "min_gvf5": min(cu.gvf(c[s]) for s in rec_ids),
                 "rho_vs_E0": " ".join(f"{c[s].corr(cri[rec_table][s], method='spearman'):.2f}" for s in rec_ids),
                 "cri_mean": " ".join(f"{c[s].mean():.2f}" for s in rec_ids)})
print("E1wd core weights (vuln, resp, risk):", np.round(w_e1wd, 3))
print(pd.DataFrame(rows).round(3).to_string(index=False))

cand = metrics[rec_table].query("n_categories >= @min_categories").index.tolist()
search_e1wd = cu.search_sets(cri_e1wd, cand, k=4, frac=top_frac).sort_values(["min_dissim", "mean_dissim"], ascending=False)
search_e1wd.to_csv(f"{out_dir}/set_search_T1g_E1wd.csv", index=False)
rank = int((search_e1wd["min_dissim"] > search_e1wd.loc[search_e1wd["set"] == tuple(rec_ids), "min_dissim"].iloc[0]).sum()) + 1
print(f"Recommended regimes under E1wd: rank {rank} of {len(search_e1wd)}")
print(describe_set(rec_table, search_e1wd.iloc[0]["set"]).round(3).to_string())
search_e1wd.head(5).round(3)

In [ ]:
# (D) Response to a change of the vulnerability model: same regime time, other table vs T1g
def regime_cri(n, k):
    v, st = vuln[n], states[n]
    wk = list(v.columns)
    if k == "E0":
        c = cri[n][reg_ids[n]]
    else:
        pw = {s: 1.0 for s in wk} if k == "E1w" else cu.week_fractions(st)
        c = cu.cri_pooled_weighted(v, base, reg_ids[n], pw)[0]
    return c.set_axis(reg_times, axis=1)

regime_cache = {(n, k): regime_cri(n, k) for n in cand_tables for k in ["E0", "E1w", "E1wd"]}
rows = []
for n in ["T1", "T2", "T2g"]:
    for i, t in enumerate(reg_times):
        r = {"change": f"T1g -> {n}", "time": t,
             "vuln_rho": vuln[rec_table][reg_ids[rec_table][i]].corr(vuln[n][reg_ids[n][i]], method="spearman")}
        for k in ["E0", "E1w", "E1wd"]:
            r[f"cri_rho_{k}"] = regime_cache[(rec_table, k)][t].corr(regime_cache[(n, k)][t], method="spearman")
        rows.append(r)
model_change = pd.DataFrame(rows)
model_change.to_csv(f"{out_dir}/model_change_response.csv", index=False)
for k in ["E0", "E1w", "E1wd"]:
    print(f"{k}: Spearman(vulnerability change, CRI change) = "
          f"{(1 - model_change.vuln_rho).corr(1 - model_change[f'cri_rho_{k}'], method='spearman'):.3f}")
model_change.round(3)

### Findings

- **Scan resolution does not matter.** Rescaling all multiplicities by one constant changes the
  denominator of every indicator's normalized entropy equally, which cancels when the weights are
  normalized. Slot counts at 15, 30 or 60 min give exactly the E1wd weights.
- **Fragmentation separates the schemes.** Splitting a state into two identical states changes the E1w
  weights (vulnerability 0.566 → 0.598) and leaves E1wd unchanged. E1w depends on how the table happens to
  partition the week; E1wd depends only on how long each activity pattern lasts.
- **Stability across tables.** Over T1, T1g, T2 and T2g the E1wd vulnerability weight stays within
  0.656–0.675 (E1w: 0.525–0.569; E0 per scenario: 0.12–0.75).
- **Discrimination.** For the T1g regimes, mean dissimilarity is 0.534 under E1wd (E1w 0.373, E0 0.277),
  with within-scenario class separation kept (GVF ≥ 0.91).
- **Response to model changes.** All schemes follow the size of a vulnerability change (Spearman 0.91–0.98
  over 12 cases). E0 damps changes in busy scenarios (11:00, T1g → T2: vulnerability 0.33, CRI 0.73);
  E1wd passes them through (0.31) and damps them at night, where vulnerability differences are small on
  the common scale. Pooled weights are non-local: a change in one period moves the weights of the week,
  so every scenario shifts slightly (08:30: CRI 0.998 with unchanged vulnerability).
- **Costs of E1wd.** Vulnerability takes ≈0.67 of the core weight (response ≈0.18, flood risk ≈0.14),
  driven by the concentrated vulnerability of the night, which is 32% of the week. The CRI moves far
  from the article's (morning rush: Spearman 0.47 with E0), and the best scenario set changes.
- **Determinism.** Re-running VERUS for a cached state reproduces it to 1e-16, so the chain
  POTIs → vulnerability → weights → CRI is deterministic for a given table and POTI snapshot.

## 9.11. Pending work

Adopted in the pipeline (notebooks 01, 06, 08 and 10): table T1g, the four regimes of section 9.8, and the
article's per-scenario entropy weights (E0). This notebook remains a sensitivity analysis and is not needed
to reproduce the pipeline.

1. **verus.** Released 1.1.1 with the `vi` carry-over fix, the `run(data_source=df)` fix and the zero-baseline
   VL normalization. Still to do: make `_apply_time_windows_to_potis` robust to two active windows of the
   same category (the merge duplicates POTIs; T1g avoids it with exclusive window ends).
2. **Entropy weighting.** Decided: E0 stays in the pipeline. E1wd (section 9.10) goes into the thesis as a
   sensitivity analysis, with its invariance properties and costs.
3. **Porto.** Validate the table, the scenario set and the CRI on Porto.
4. **Night scenario.** With few active POTIs, the KMeans cluster boundaries show up as polygons in the
   21:00 map. Check whether this comes from per-cluster computation and smoothing in verus.
5. **T2g values.** T2g separates slightly more (min_dissim 0.142 vs 0.132) but relies on new `vi` values
   (weekend malls 0.7, attractions until 20:00). Adopting it is a domain decision.
6. **Housekeeping.** Clear this notebook's outputs before committing (the maps make it about 6 MB).